In [1]:
# clone repo
!git clone https://github.com/eth-bmai-fs26/project3-agentic-tax-filler.git

import sys
sys.path.insert(0, "/content/project3-agentic-tax-filler")

Cloning into 'project3-agentic-tax-filler'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 31 (delta 0), reused 31 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 3.54 MiB | 45.34 MiB/s, done.


In [19]:
# imports and server
from mcp_server.server import MCPServer
from mcp_server.bridges.mock import MockBridge

server = MCPServer(
    persona_folder="/content/project3-agentic-tax-filler/personas/anna_meier",
    guides_folder="/content/project3-agentic-tax-filler/guides",
    bridge=MockBridge()
)
print("server ready!")

server ready!


In [40]:
#dummy file for Anna Meier
import os

#1. folder path
folder_path = "/content/project3-agentic-tax-filler/personas/anna_meier"
os.makedirs(folder_path, exist_ok=True)

#2. dummy tax document
dummy_lohnausweis = """
LOHNAUSWEIS (Salary Statement) 2025
Name: Anna Meier
Employer: Google Zurich
Role: Software Engineer

Gross Salary (Bruttolohn): 95,000 CHF
AHV Contributions: 5,000 CHF
Transit Pass (ZVV): 800 CHF
"""

# 3. save in a folder
file_path = os.path.join(folder_path, "lohnausweis_2025.txt")
with open(file_path, "w") as f:
    f.write(dummy_lohnausweis)

print(f"Created dummy document at: {file_path}")

Created dummy document at: /content/project3-agentic-tax-filler/personas/anna_meier/lohnausweis_2025.txt


In [ ]:
# llm set up — supports OpenAI (via ETH proxy) or Gemini
import os
import json
from openai import OpenAI

# ── Choose your provider: "openai" or "gemini" ──
LLM_PROVIDER = "gemini"  # <-- change this to "openai" to use the ETH proxy

if LLM_PROVIDER == "openai":
    os.environ["OPENAI_API_KEY"] = "..."  # paste your ETH proxy key
    client = OpenAI(
        api_key=os.environ["OPENAI_API_KEY"],
        base_url="https://litellm.sph-prod.ethz.ch/v1"  # ETH proxy
    )
    LLM_MODEL = "anthropic/claude-sonnet-4-5"

elif LLM_PROVIDER == "gemini":
    os.environ["GEMINI_API_KEY"] = "..."  # paste your Gemini API key
    client = OpenAI(
        api_key=os.environ["GEMINI_API_KEY"],
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/"  # Gemini OpenAI-compatible endpoint
    )
    LLM_MODEL = "gemini-2.0-flash"

else:
    raise ValueError(f"Unknown LLM_PROVIDER: {LLM_PROVIDER!r}. Use 'openai' or 'gemini'.")

print(f"Using {LLM_PROVIDER} with model {LLM_MODEL}")

SYSTEM_PROMPT = """
You are an expert Swiss tax accountant AI autonomously filing a Zurich tax return.

You have access to these tools:
- list_documents: list files in the persona folder. Args: {}
- read_document: read a file. Args: {"filepath": "filename.pdf"}
- scan_page: see the current form page and its fields. Args: {}
- fill_field: fill a form field. Args: {"locator": "field-id", "value": "value"}
- click_element: click a button to navigate. Args: {"locator": "btn-id"}
- list_guides: see available tax guides. Args: {}
- fetch_guide: read a tax guide. Args: {"url": "path/to/guide"}
- ask_user: ask the taxpayer a question (USE SPARINGLY). Args: {"question": "..."}
- submit_form: submit when fully complete. Args: {}

## Core Strategy
1. KNOWLEDGE FIRST: Always list and read ALL documents before touching the form.
2. PROACTIVE DEDUCTIONS (Guides): You must actively search for tax rules!
   - If the client has children, commutes, works from home, pays alimony, or is a freelancer, DO NOT guess the deduction limits. Call `list_guides`, find the relevant rule, and call `fetch_guide`.
3. RESOLVE AMBIGUITY (Ask User): If a receipt is blurry, an amount is cut off, or a document is ambiguous (e.g., is the alimony for the spouse or the child?), you MUST call `ask_user` to get clarification. Do not make assumptions.
4. NAVIGATE & FILL: Scan the page, fill all applicable fields based on rules and documents, then click to the next page.

## Response Format
You must respond with ONLY a valid JSON object.
Crucially, you must include a "thought" field before the tool call to explain your reasoning, calculations, or why you are searching a guide.
Example:
{
  "thought": "I see a ZVV transit pass for 800 CHF. I need to check the tax guides to see if commuting costs are fully deductible or if there is a cap.",
  "tool": "list_guides",
  "args": {}
}
"""

In [21]:
def parse_llm_response(text):
    """Extract JSON tool call from LLM response."""
    text = text.strip()
    # Strip markdown code fences if present
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        print(f"  [WARN] Could not parse LLM response: {text}")
        # Safe fallback
        return {"tool": "scan_page", "args": {}}

In [39]:
import json

def think (server, client, system_prompt, max_steps = 50):
  conversation_history = []
  #current step
  step=0

  #state tracking
  documents_read=[]
  guides_read=[]
  questions_asked=[]
  has_listed_documents=False

  while step<max_steps:
    step+=1
    print(f"\n--- Step {step} ---")

    #1. current UI state
    current_ui = server.scan_page()

    #2.build context for the LLM
    if not has_listed_documents:
      user_message = """You have just started. You do NOT know who the client is yet. Action Required: Call `list_documents` to see what is in their folder."""
    else:
      #we feed the current state to the LLM so it doesn't repeat actions
      user_message = f"""[CURRENT STATE]
      Documents read: {documents_read}
      Guides read: {guides_read}
      Questions asked: {questions_asked}

      [CURRENT UI PAGE]
      {json.dumps(current_ui, indent=2)}

      Based on the documents and rules, what is your next action? (Remember to use the "thought" key to reason out your math or strategy first)."""

    # --- THE FIX: Un-indented everything below this line ---

    conversation_history.append({
                "role": "user",
                "content": user_message
    })

    #3. think
    response = client.chat.completions.create(
        model=LLM_MODEL,
        max_tokens=500,
        messages=[{"role": "system", "content": system_prompt}]+conversation_history
        )
    llm_text=response.choices[0].message.content
    conversation_history.append({"role": "assistant", "content": llm_text})

    #4. parse action
    action = parse_llm_response(llm_text)
    print(f"🧠  Thought: {action.get('thought', 'No thought provided.')}")
    print(f"🤖  Action:  {action['tool']}({action.get('args', {})})")

    #5. stop if 'done'
    if action['tool'] in ("done", "submit_form"):
      result=server.submit_form()
      print(f" Submitted! Result: {result}")
      break

    #6. execute tool
    try:
      result=server.execute(action)
      #print the shorten version in colab
      print(f" result: {str(result)[:200]}...")
    except Exception as e:
      print(f"  [ERROR] {e}")
      # Need to define result here so step 8 doesn't crash on an error
      result = {"error": str(e)}

    #7. track state to prevent loops
    if action['tool'] == "list_documents":
      has_listed_documents=True
    elif action['tool'] == "read_document":
      documents_read.append(action["args"].get("filepath"))
    elif action['tool'] == "fetch_guide":
      guides_read.append(action["args"].get("url"))
    elif action['tool'] == "ask_user":
      questions_asked.append(action["args"].get("question"))

    #8. concatenate the results
    conversation_history.append({
      "role": "user",
      "content": f"Tool execution result:\n{json.dumps(result, indent=2, default=str)}"
    })

  else:
    print(f"WARNING: max steps ({max_steps}) reached without submitting.")

In [42]:
print("Starting autonomous tax agent...")
#re-initialize your server right before running the loop.
server = MCPServer(
    persona_folder="/content/project3-agentic-tax-filler/personas/anna_meier",
    guides_folder="/content/project3-agentic-tax-filler/guides",
    bridge=MockBridge()
)
think(server, client, SYSTEM_PROMPT, max_steps=50)
print("Done!")

Starting autonomous tax agent...

--- Step 1 ---
🧠  Thought: I'm starting the tax return process. The first step is to discover who the client is and what documents they have provided. I need to list all documents in the persona folder to understand what information is available before I begin filling out any forms.
🤖  Action:  list_documents({})
 result: ['lohnausweis_2025.txt']...

--- Step 2 ---
🧠  Thought: I can see there is one document available: 'lohnausweis_2025.txt' which is a salary statement. Before I can proceed with logging into the tax system, I need to read this document to understand the client's information and income details. This will give me the taxpayer's name and other essential details needed for the tax return.
🤖  Action:  read_document({'filepath': 'lohnausweis_2025.txt'})
 result: {'filepath': 'lohnausweis_2025.txt', 'type': 'text', 'content': '\nLOHNAUSWEIS (Salary Statement) 2025\nName: Anna Meier\nEmployer: Google Zurich\nRole: Software Engineer\n\nGross Sa

In [27]:
# test
# use tools
current_ui = server.scan_page()
print(json.dumps(current_ui, indent=2))
server.scan_page()
server.fill_field(locator="field-income-employment-bruttolohn", value=95000)
ui_after_filling_income = server.scan_page()
print(json.dumps(ui_after_filling_income, indent=2))

server.click_element(locator="btn-nav-income")
ui_after_nav = server.scan_page()
print(json.dumps(ui_after_nav, indent=2))

server.submit_form()
server.list_documents()
server.ask_user("What is your filing status?")

{
  "page_name": "login",
  "validation_errors": [],
  "elements": [
    {
      "type": "input",
      "locator": "field-login-username",
      "label": "Username",
      "value": "",
      "required": true
    },
    {
      "type": "input",
      "locator": "field-login-password",
      "label": "Password",
      "value": "",
      "required": true
    },
    {
      "type": "button",
      "locator": "btn-login",
      "label": "Login"
    }
  ]
}
{
  "page_name": "login",
  "validation_errors": [],
  "elements": [
    {
      "type": "input",
      "locator": "field-login-username",
      "label": "Username",
      "value": "",
      "required": true
    },
    {
      "type": "input",
      "locator": "field-login-password",
      "label": "Password",
      "value": "",
      "required": true
    },
    {
      "type": "button",
      "locator": "btn-login",
      "label": "Login"
    }
  ]
}
{
  "page_name": "income",
  "validation_errors": [],
  "elements": [
    {
      "type"

{'question': 'What is your filing status?',
 'answer': "I'm not sure about that. You might need to check my documents.",
 'timestamp': '2026-03-11T10:24:47.699062+00:00'}